# Robustness: Train/Valid/Test Splits

比較不同時間切分下 Logistic baseline (有 GICS, Platt 校準) 的表現。


In [1]:
!pip install -U pandas numpy scikit-learn shap matplotlib openpyxl



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [2]:
import os, pandas as pd
TCRI_PATH = 'tcri.csv'
FIN_PATH  = 'ratios_filled_with_gics_category.csv'
ROOT_OUT  = 'outputs/robustness_splits'
os.makedirs(ROOT_OUT, exist_ok=True)
MERGED_PATH = os.path.join(ROOT_OUT, 'merged.csv')

!python merge_tcri_and_ratios.py \
  --tcri "$TCRI_PATH" \
  --ratios  "$FIN_PATH" \
  --ratios-date-format "%Y/%m" \
  --dedup-ratios \
  --out "$MERGED_PATH"

merged = pd.read_csv(MERGED_PATH, parse_dates=['mdate'])
merged.head()


[INFO] Dropped 88 duplicate rows from ratios data based on (id,date)
[INFO] Merged shape: (6383, 19). Saved to /Users/chieh.1227/repo/ML/outputs/robustness_splits/merged.csv


,coid,mdate,tcri,scr,xcdt,company,WorkingCapital_TotalAssets,RetainedEarnings_TotalAssets,CashFlow_TotalDebt,TotalDebt_TotalAssets,CurrentRatio,stock_prefix,WorkingCapital_TotalAssets_miss,RetainedEarnings_TotalAssets_miss,CashFlow_TotalDebt_miss,TotalDebt_TotalAssets_miss,CurrentRatio_miss,產業別,GICS_Category
0,1103,2014-12-01,6,400.0,NaN,1103 嘉泥,0.248834,0.222521,0.031686,0.355758,1.573942,11,0,0,0,0,0,水泥工業,9
1,1103,2015-12-01,6,432.0,NaN,1103 嘉泥,0.288490,0.266805,0.022494,0.396789,2.079358,11,0,0,0,0,0,水泥工業,9
2,1103,2016-12-01,6,371.0,NaN,1103 嘉泥,0.251213,0.276883,0.039433,0.332750,1.751954,11,0,0,0,0,0,水泥工業,9
3,1103,2017-12-01,6,330.0,NaN,1103 嘉泥,0.214906,0.277971,0.043575,0.277508,1.595351,11,0,0,0,0,0,水泥工業,9
4,1103,2018-12-01,6,331.0,NaN,1103 嘉泥,0.222120,0.264199,-0.055883,0.297109,1.501782,11,0,0,0,0,0,水泥工業,9


In [3]:
import os, json, pandas as pd, subprocess

ROOT_OUT  = 'outputs/robustness_splits'
MERGED_PATH = os.path.join(ROOT_OUT, 'merged.csv')

SPLITS = [
    {
        'name': 'baseline',
        'train_start': '2014-01-01', 'train_end': '2021-12-31',
        'valid_start': '2022-01-01', 'valid_end': '2022-12-31',
        'test_start':  '2023-01-01', 'test_end':  '2023-12-31',
    },
    {
        'name': 'shorter_train',
        'train_start': '2016-01-01', 'train_end': '2020-12-31',
        'valid_start': '2021-01-01', 'valid_end': '2021-12-31',
        'test_start':  '2022-01-01', 'test_end':  '2023-12-31',
    },
]

results = {}
for cfg in SPLITS:
    name = cfg['name']
    outdir = os.path.join(ROOT_OUT, name)
    os.makedirs(outdir, exist_ok=True)
    print('=== Running split:', name, '->', outdir, '===')
    cmd = [
        'python', 'tcri_baseline_logit.py',
        '--csv', MERGED_PATH,
        '--gics-col', 'GICS_Category',
        '--tau', '7',
        '--train-start', cfg['train_start'], '--train-end', cfg['train_end'],
        '--valid-start', cfg['valid_start'], '--valid-end', cfg['valid_end'],
        '--test-start',  cfg['test_start'],  '--test-end',  cfg['test_end'],
        '--outdir', outdir,
    ]
    subprocess.run(cmd, check=True)
    with open(os.path.join(outdir, 'metrics_summary.json'), 'r', encoding='utf-8') as f:
        results[name] = json.load(f)
print('Done.')


=== Running split: baseline -> outputs/robustness_splits/baseline ===
[INFO] Selected C=10.0 by PR-AUC on validation (0.7484)
[INFO] Thresholds on validation: t_f1=0.5101, t_at_P>=0.50 = 0.2480


/Users/chieh.1227/repo/ML/.venv/lib/python3.12/site-packages/sklearn/calibration.py:330: FutureWarning: The `cv='prefit'` option is deprecated in 1.6 and will be removed in 1.8. You can use CalibratedClassifierCV(FrozenEstimator(estimator)) instead.
  warnings.warn(


SHAP plots saved under outputs/plots: shap_summary_baseline_logit.png, shap_bar_baseline_logit.png

=== Test Overall Metrics (threshold = t_f1 on valid) ===
[raw] PR-AUC=0.6542  ROC-AUC=0.8018  F1=0.6130  Recall@P>=0.50=0.7989  Brier=0.1699  ECE=0.1121
[platt] PR-AUC=0.6542  ROC-AUC=0.8018  F1=0.5977  Recall@P>=0.50=0.7989  Brier=0.1575  ECE=0.0633

Top 10 feature weights by |coefficient|:
                          feature  coefficient  abs_coefficient
             cat__GICS_Category_7    -2.960184         2.960184
num__RetainedEarnings_TotalAssets    -1.844111         1.844111
            cat__GICS_Category_10     1.700931         1.700931
          num__CashFlow_TotalDebt    -0.527946         0.527946
             cat__GICS_Category_9    -0.507665         0.507665
  num__WorkingCapital_TotalAssets    -0.437205         0.437205
             cat__GICS_Category_8     0.411531         0.411531
            cat__GICS_Category_11     0.360624         0.360624
             cat__GICS_Category

/Users/chieh.1227/repo/ML/.venv/lib/python3.12/site-packages/sklearn/calibration.py:330: FutureWarning: The `cv='prefit'` option is deprecated in 1.6 and will be removed in 1.8. You can use CalibratedClassifierCV(FrozenEstimator(estimator)) instead.
  warnings.warn(


SHAP plots saved under outputs/plots: shap_summary_baseline_logit.png, shap_bar_baseline_logit.png

=== Test Overall Metrics (threshold = t_f1 on valid) ===
[raw] PR-AUC=0.7012  ROC-AUC=0.8120  F1=0.6516  Recall@P>=0.50=0.8376  Brier=0.1658  ECE=0.0889
[platt] PR-AUC=0.7012  ROC-AUC=0.8120  F1=0.6107  Recall@P>=0.50=0.8376  Brier=0.1563  ECE=0.0367

Top 10 feature weights by |coefficient|:
                          feature  coefficient  abs_coefficient
num__RetainedEarnings_TotalAssets    -1.850475         1.850475
            cat__GICS_Category_10     1.590500         1.590500
             cat__GICS_Category_7    -1.221459         1.221459
             cat__GICS_Category_9    -0.596605         0.596605
          num__CashFlow_TotalDebt    -0.462828         0.462828
  num__WorkingCapital_TotalAssets    -0.394386         0.394386
             cat__GICS_Category_3    -0.234802         0.234802
             cat__GICS_Category_5     0.160347         0.160347
             cat__GICS_Category

In [4]:
import pandas as pd

rows = []
for split_name, metrics in results.items():
    for name in ['raw', 'platt']:
        m = metrics[name]
        rows.append({
            'split': split_name,
            'calibration': name,
            'pr_auc': m.get('pr_auc'),
            'roc_auc': m.get('roc_auc'),
            'f1': m.get('f1'),
            'precision': m.get('precision'),
            'recall': m.get('recall'),
            'brier': m.get('brier'),
            'ece': m.get('ece'),
        })
df_splits = pd.DataFrame(rows)
df_splits.sort_values(['calibration','split'])


,split,calibration,pr_auc,roc_auc,f1,precision,recall,brier,ece
1,baseline,platt,0.654198,0.801778,0.597701,0.615385,0.581006,0.157525,0.063335
3,shorter_train,platt,0.701238,0.812002,0.610729,0.666667,0.563452,0.156331,0.036697
0,baseline,raw,0.654198,0.801778,0.612987,0.572816,0.659218,0.169858,0.112107
2,shorter_train,raw,0.701238,0.812002,0.651551,0.614865,0.692893,0.165763,0.088892
